# MILCS: win95pts Example

This notebook runs the bundled 5,000-sample-per-environment linear Gaussian example. Evaluation covers only the original target V63 and both endpoints of its incident edges. Select the Python environment containing requirements.txt dependencies.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "MILCS.py").is_file():
    raise RuntimeError("Open this notebook from the repository root.")
print("Python:", sys.version.split()[0])
print("Example configuration:")
print((ROOT / "example" / "config.json").read_text())


Python: 3.10.18
Example configuration:
{
  "seed": 20261024,
  "samples_per_environment": 5000,
  "latent_count": 3,
  "environment_count": 3,
  "intervention_size": 3,
  "target": "V63"
}



## Continuous linear Gaussian data

The dataset has 73 observed variables, three latent variables, and three hard-intervention environments with 5,000 samples each. The following cell calls the same command-line entry point as `python MILCS.py`.


In [2]:
def run_milcs(ci_type):
    completed = subprocess.run(
        [sys.executable, "-B", str(ROOT / "MILCS.py"),
         "--target", "V63", "--ci-type", ci_type],
        cwd=ROOT,
        env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
        capture_output=True, text=True, check=True,
    )
    print(completed.stdout)
    return completed.stdout

continuous_output = run_milcs("Fisher_Z")


 Loaded 3 interventional DAGs
Iteration guard: total_nodes=76, max_processed_nodes=15
MB learner: gaussian_MB_learn, data_type=n/a, max_k=n/a, CI=Fisher_Z
===== Start =====
Target node: V63
alpha: 0.01
WaitList policy: all_local
 Building the initial global causal structure G0
------------------------------------
 Processing node V31
------------------------------------
 Processing node V32
------------------------------------
 Processing node V34
------------------------------------
 Processing node V35
------------------------------------
 Processing node V36
------------------------------------
 Processing node V51
 Stop: target V63 has no unresolved circle endpoints
TRAVERSAL_STATS {"coverage": 0.0958904109589041, "exit_reason": "target V63 has no unresolved circle endpoints", "processed_observed": 7, "total_observed": 73, "traversed_all": false, "unprocessed_nodes": ["V0", "V1", "V10", "V11", "V12", "V13", "V14", "V15", "V16", "V17", "V18", "V19", "V2", "V20", "V21", "V22", "V24",

## Oracle comparison

Oracle mode uses the true intervention DAGs for d-separation rather than sampled observations. It still learns and evaluates the local target structure.


In [3]:
oracle_output = run_milcs("D_sep")


 Loaded 3 interventional DAGs
Iteration guard: total_nodes=76, max_processed_nodes=15
MB learner: TC_learn, data_type=n/a, max_k=n/a, CI=D_sep
===== Start =====
Target node: V63
alpha: 0.01
WaitList policy: all_local
 Building the initial global causal structure G0
------------------------------------
 Processing node V51
------------------------------------
 Processing node V35
------------------------------------
 Processing node V36
------------------------------------
 Processing node V34
------------------------------------
 Processing node V67
------------------------------------
 Processing node V10
------------------------------------
 Processing node V7
------------------------------------
 Processing node V9
------------------------------------
 Processing node V2
------------------------------------
 Processing node V8
------------------------------------
 Processing node V20
------------------------------------
 Processing node V57
------------------------------------
 Proc

The current scoring convention permits a circle or a tail at a causal source endpoint when direct causation and latent confounding coexist. A perfect Mark score is agreement with that convention, not proof that every endpoint has been fully oriented. See README.md for details.
